# LeetCode 1545 — Find Kth Bit in Nth Binary String bằng Python

Link đề: https://leetcode.com/problems/find-kth-bit-in-nth-binary-string/ · Độ khó: **Medium** · Chủ đề: String, Recursion, Simulation

Skeleton LeetCode:

```python
class Solution:
    def findKthBit(self, n: int, k: int) -> str:
        ...
```

## 1. Mục tiêu học

1. Hiểu định nghĩa đệ quy `Sᵢ = Sᵢ₋₁ + "1" + reverse(invert(Sᵢ₋₁))` và tự sinh chuỗi bằng mô phỏng (cách theo gợi ý của đề).
2. Nắm cấu trúc **đối xứng gương**: `Sₙ` dài `2ⁿ − 1`, giữa là `"1"` ở vị trí `2ⁿ⁻¹`, nửa phải là ảnh gương **đảo bit** của nửa trái.
3. Viết lời giải **đệ quy theo vị trí** `O(n)` không cần sinh chuỗi: `k` ở nửa phải ⇒ ánh xạ về `2ⁿ − k` ở `Sₙ₋₁` và đảo bit.
4. Khử đệ quy thành vòng lặp với **biến đếm số lần đảo**, và (nâng cao) tìm công thức `O(1)` bằng thao tác bit.
5. Tránh các bẫy: chỉ số gương lệch một, quên đảo bit, quên đảo ngược, dùng `replace` sai để đảo bit, trả về `int` thay vì `str`.

## 2. Đề bài & ví dụ

Cho hai số nguyên dương `n` và `k`. Chuỗi nhị phân `Sₙ` được tạo như sau:

- `S₁ = "0"`
- `Sᵢ = Sᵢ₋₁ + "1" + reverse(invert(Sᵢ₋₁))` với `i > 1`,

trong đó `+` là nối chuỗi, `reverse(x)` đảo ngược chuỗi, `invert(x)` đảo mọi bit (0 ↔ 1).

Bốn chuỗi đầu: `S₁ = "0"`, `S₂ = "011"`, `S₃ = "0111001"`, `S₄ = "011100110110001"`.

Trả về **bit thứ `k`** (đếm từ 1) của `Sₙ` dưới dạng chuỗi `"0"` hoặc `"1"`. Bảo đảm `k` hợp lệ.

**Ví dụ 1:** `n = 3, k = 1` → `"0"` (`S₃ = "0111001"`).

**Ví dụ 2:** `n = 4, k = 11` → `"1"` (`S₄ = "011100110110001"`, bit thứ 11 là `"1"`).

**Ràng buộc:** `1 ≤ n ≤ 20`, `1 ≤ k ≤ 2ⁿ − 1`.

In [1]:
import random
import time
from functools import lru_cache
from typing import List

examples = [(3, 1, "0"), (4, 11, "1")]
for n, k, want in examples:
    print(f"n={n}, k={k:>2} -> mong đợi {want!r}")

n=3, k= 1 -> mong đợi '0'
n=4, k=11 -> mong đợi '1'


Hai ví dụ thường gặp khác (kiểm chứng bằng mô phỏng ở mục 5; nên đối chiếu lại đề gốc): `n = 1, k = 1` → `"0"`; `n = 2, k = 3` → `"1"`.

In [2]:
more = [(1, 1, "0"), (2, 3, "1")]
print(more)

[(1, 1, '0'), (2, 3, '1')]


## 3. Kiến thức nền

### 3.1 Đảo ngược chuỗi bằng cắt lát

`x[::-1]` tạo chuỗi đảo ngược (`O(len)`).

In [3]:
print("0111"[::-1], "abc"[::-1])

1110 cba


### 3.2 Đảo bit một chuỗi `'0'/'1'`: `str.translate`

`str.maketrans("01", "10")` tạo bảng thay đồng thời `0→1`, `1→0`. **Không** dùng hai lần `replace` liên tiếp (lỗi 5 ở mục 13).

In [4]:
INV = str.maketrans("01", "10")
print("0111001".translate(INV))
print("".join("1" if c == "0" else "0" for c in "0111001"))

1000110
1000110


### 3.3 Độ dài và vị trí giữa

`len(Sₙ) = 2·len(Sₙ₋₁) + 1`, với `len(S₁) = 1` ⇒ `len(Sₙ) = 2ⁿ − 1`. Vị trí giữa (1-indexed) là `2ⁿ⁻¹`. Trong Python: `1 << (n - 1)` = `2ⁿ⁻¹`.

In [5]:
for n in range(1, 6):
    print(f"n={n}: độ dài {(1 << n) - 1:>2}, vị trí giữa {1 << (n - 1):>2}")

n=1: độ dài  1, vị trí giữa  1
n=2: độ dài  3, vị trí giữa  2
n=3: độ dài  7, vị trí giữa  4
n=4: độ dài 15, vị trí giữa  8
n=5: độ dài 31, vị trí giữa 16


### 3.4 Đệ quy và khử đệ quy

Hàm đệ quy gọi lại chính nó với bài toán **nhỏ hơn** (`n − 1`), dừng ở trường hợp cơ sở (`n = 1`). Độ sâu tối đa 20 — an toàn. Có thể chuyển thành vòng lặp `while` khi mỗi lần chỉ gọi đệ quy **một** lần (đệ quy đuôi + một phép biến đổi kết quả).

In [6]:
def length(n):
    return 1 if n == 1 else 2 * length(n - 1) + 1

def length_iter(n):
    L = 1
    for _ in range(n - 1):
        L = 2 * L + 1
    return L

print([length(n) for n in range(1, 8)], [length_iter(n) for n in range(1, 8)])

[1, 3, 7, 15, 31, 63, 127] [1, 3, 7, 15, 31, 63, 127]


### 3.5 Bit thấp nhất: `k & -k`

`k & -k` giữ lại **bit 1 thấp nhất** của `k` (ví dụ `12 = 1100₂` → `4`). Số mũ `a` = `(k & -k).bit_length() − 1`, phần lẻ `m = k >> a`, và `k = 2ᵃ · m`. Dùng ở cách 4.

In [7]:
for k in [1, 6, 12, 20, 7]:
    low = k & -k
    a = low.bit_length() - 1
    print(f"k={k:>2} = 2^{a} * {k >> a}  (bit thấp nhất = {low})")

k= 1 = 2^0 * 1  (bit thấp nhất = 1)
k= 6 = 2^1 * 3  (bit thấp nhất = 2)
k=12 = 2^2 * 3  (bit thấp nhất = 4)
k=20 = 2^2 * 5  (bit thấp nhất = 4)
k= 7 = 2^0 * 7  (bit thấp nhất = 1)


## 4. Ý tưởng nền tảng / trực giác

Nhìn `S₄ = 011100110110001` (15 ký tự) chia làm ba phần:

```
vị trí : 1 2 3 4 5 6 7 | 8 | 9 10 11 12 13 14 15
S₄     : 0 1 1 1 0 0 1 | 1 | 0  1  1  0  0  0  1
         ---- S₃ ----    giữa  -- reverse(invert(S₃)) --
```

- Vị trí 8 (giữa) luôn là `"1"`.
- Vị trí `k = 11` ở nửa phải. Đi "soi gương" qua vị trí giữa: vị trí đối xứng là `16 − 11 = 5` (tổng hai vị trí đối xứng = `2ⁿ = 16`). Bit ở vị trí 5 của `S₃` là `"0"`, đảo bit ⇒ `"1"` ✔.

Vậy ta không cần sinh chuỗi: chỉ cần "đi" từ `Sₙ` xuống `S₁`, mỗi bước hoặc dừng (giữa), hoặc giữ nguyên `k` (nửa trái), hoặc soi gương + nhớ đảo bit (nửa phải).

In [8]:
S3 = "0111001"
S4 = S3 + "1" + S3.translate(INV)[::-1]
print("S4 =", S4)
k = 11
print(f"bit thứ {k} của S4 = {S4[k - 1]} | vị trí gương = {16 - k} | S3[{16 - k}] = {S3[16 - k - 1]} -> đảo = {'1' if S3[16 - k - 1] == '0' else '0'}")

S4 = 011100110110001
bit thứ 11 của S4 = 1 | vị trí gương = 5 | S3[5] = 0 -> đảo = 1


## 5. Cách 1 — Brute Force / baseline: sinh chuỗi `Sₙ` (theo gợi ý của đề)

Vì `n ≤ 20`, `Sₙ` dài tối đa `2²⁰ − 1 = 1 048 575` ký tự — sinh được. Lặp `n − 1` lần công thức, rồi lấy ký tự `k − 1`.

In [9]:
def build_S(n):
    s = "0"
    for _ in range(n - 1):
        s = s + "1" + s.translate(INV)[::-1]
    return s

def brute(n, k):
    return build_S(n)[k - 1]

for i in range(1, 5):
    print(f"S{i} = {build_S(i)}")
for n, k, want in examples + more:
    print(n, k, "->", brute(n, k), "mong đợi", want)

S1 = 0
S2 = 011
S3 = 0111001
S4 = 011100110110001
3 1 -> 0 mong đợi 0
4 11 -> 1 mong đợi 1
1 1 -> 0 mong đợi 0
2 3 -> 1 mong đợi 1


**Độ phức tạp:** tổng độ dài các chuỗi sinh ra `≈ 2ⁿ⁺¹` ⇒ Time `O(2ⁿ)`, Space `O(2ⁿ)`. Với `n = 20` khoảng 2 triệu ký tự — chấp nhận được trên LeetCode, nhưng nếu `n` lên 60 (biến thể) thì bất khả. Đo thử:

In [10]:
t0 = time.perf_counter()
s20 = build_S(20)
print(f"len(S20) = {len(s20)} ({time.perf_counter() - t0:.3f}s), bit cuối = {s20[-1]}")

len(S20) = 1048575 (0.004s), bit cuối = 1


**Nút thắt:** ta sinh hàng triệu ký tự chỉ để đọc **một** ký tự. Cấu trúc gương cho phép nhảy thẳng tới câu trả lời.

## 6. Ý tưởng thuật toán / bước đột phá

Gọi `f(n, k)` là bit thứ `k` của `Sₙ`, `mid = 2ⁿ⁻¹`.

**Quy tắc "nếu… thì…":**
- **Nếu** `n = 1` **thì** `"0"`.
- **Nếu** `k = mid` **thì** `"1"`.
- **Nếu** `k < mid` **thì** `f(n − 1, k)` (nửa trái chính là `Sₙ₋₁`).
- **Nếu** `k > mid` **thì** `invert(f(n − 1, 2ⁿ − k))`.

**Vì sao `2ⁿ − k`?** Nửa phải bắt đầu ở `mid + 1`. Ký tự ở vị trí `k` của `Sₙ` là ký tự thứ `j = k − mid` của `reverse(invert(Sₙ₋₁))`, tức ký tự thứ `L − j + 1` của `invert(Sₙ₋₁)` với `L = 2ⁿ⁻¹ − 1`. Tính: `L − (k − mid) + 1 = (2ⁿ⁻¹ − 1) − k + 2ⁿ⁻¹ + 1 = 2ⁿ − k`.

**Invariant (dạng lặp):** tại mỗi bước, đáp án = `f(n_hiện_tại, k_hiện_tại)` đảo bit `flips` lần.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy theo vị trí, `O(n)`

Viết đúng 4 quy tắc trên. Mỗi lời gọi giảm `n` đi 1 ⇒ tối đa `n` lời gọi.

In [11]:
def rec(n, k):
    if n == 1:
        return "0"
    mid = 1 << (n - 1)
    if k == mid:
        return "1"
    if k < mid:
        return rec(n - 1, k)
    bit = rec(n - 1, (1 << n) - k)          # vị trí gương trong S(n-1)
    return "1" if bit == "0" else "0"       # đảo bit

for n, k, want in examples + more:
    print(n, k, "->", rec(n, k), "mong đợi", want)

3 1 -> 0 mong đợi 0
4 11 -> 1 mong đợi 1
1 1 -> 0 mong đợi 0
2 3 -> 1 mong đợi 1


In [12]:
class SolutionRec:
    def findKthBit(self, n: int, k: int) -> str:
        if n == 1:
            return "0"
        mid = 1 << (n - 1)
        if k == mid:
            return "1"
        if k < mid:
            return self.findKthBit(n - 1, k)
        return "1" if self.findKthBit(n - 1, (1 << n) - k) == "0" else "0"

print([SolutionRec().findKthBit(n, k) for n, k, _ in examples + more])

['0', '1', '0', '1']


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho ngăn xếp đệ quy.

### Cách 3 — Vòng lặp với biến đếm số lần đảo, `O(n)` / `O(1)`

Mỗi lần "soi gương" ta chưa đảo ngay mà **ghi nhớ** `flip ^= 1`. Khi dừng (gặp giữa → bit gốc `1`, hoặc tới `n = 1` → bit gốc `0`), áp dụng `flip` một lần.

In [13]:
def iterative(n, k):
    flip = 0
    while n > 1:
        mid = 1 << (n - 1)
        if k == mid:
            return str(1 ^ flip)
        if k > mid:
            k = (1 << n) - k
            flip ^= 1
        n -= 1
    return str(0 ^ flip)                    # S1 = "0"

for n, k, want in examples + more:
    print(n, k, "->", iterative(n, k), "mong đợi", want)

3 1 -> 0 mong đợi 0
4 11 -> 1 mong đợi 1
1 1 -> 0 mong đợi 0
2 3 -> 1 mong đợi 1


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 4 — Công thức `O(1)` bằng thao tác bit (nâng cao)

Viết `k = 2ᵃ · m` với `m` lẻ. Khi đó:

- **Nếu** `k` lẻ (`a = 0`) **thì** bit = 1 ⇔ `m mod 4 = 3`.
- **Nếu** `k` chẵn (`a > 0`) **thì** bit = 1 ⇔ `m mod 4 = 1`.

Gọn lại: `bit = ((m >> 1) & 1) XOR (a > 0)`. Đặc biệt kết quả **không phụ thuộc `n`** — vì `Sₙ` là tiền tố của `Sₙ₊₁`.

*Phác thảo chứng minh (quy nạp theo bước soi gương):* vị trí giữa `k = 2ⁿ⁻¹` có `m = 1`, `a = n − 1`: công thức cho `1` (khi `n > 1`) và `0` khi `n = 1` ✔. Nửa trái giữ nguyên `k` ✔. Nửa phải: `k' = 2ⁿ − k = 2ᵃ(2ⁿ⁻ᵃ − m)`, cùng `a`, và vì `n − a ≥ 2` nên `m' = 2ⁿ⁻ᵃ − m ≡ −m (mod 4)` ⇒ `1 ↔ 3` hoán đổi ⇒ công thức cho bit **đảo** — khớp quy tắc đảo bit.

In [14]:
def formula(n, k):
    a = (k & -k).bit_length() - 1
    m = k >> a
    return str(((m >> 1) & 1) ^ (1 if a > 0 else 0))

s12 = build_S(12)
print("Công thức khớp mọi vị trí của S12:", all(formula(12, k) == s12[k - 1] for k in range(1, len(s12) + 1)))
for n, k, want in examples + more:
    print(n, k, "->", formula(n, k), "mong đợi", want)

Công thức khớp mọi vị trí của S12: True
3 1 -> 0 mong đợi 0
4 11 -> 1 mong đợi 1
1 1 -> 0 mong đợi 0
2 3 -> 1 mong đợi 1


Bảng minh hoạ công thức trên 15 vị trí của `S₄` — để thấy quy luật "lẻ: `m mod 4 = 3` cho 1; chẵn: `m mod 4 = 1` cho 1":

In [15]:
S4 = build_S(4)
print(f"{'k':>3} | {'nhị phân':>8} | {'a':>2} | {'m':>3} | m%4 | công thức | S4[k]")
for k in range(1, 16):
    a = (k & -k).bit_length() - 1
    m = k >> a
    print(f"{k:>3} | {k:>8b} | {a:>2} | {m:>3} | {m % 4:>3} | {formula(4, k):>9} | {S4[k - 1]:>5}")

  k | nhị phân |  a |   m | m%4 | công thức | S4[k]
  1 |        1 |  0 |   1 |   1 |         0 |     0
  2 |       10 |  1 |   1 |   1 |         1 |     1
  3 |       11 |  0 |   3 |   3 |         1 |     1
  4 |      100 |  2 |   1 |   1 |         1 |     1
  5 |      101 |  0 |   5 |   1 |         0 |     0
  6 |      110 |  1 |   3 |   3 |         0 |     0
  7 |      111 |  0 |   7 |   3 |         1 |     1
  8 |     1000 |  3 |   1 |   1 |         1 |     1
  9 |     1001 |  0 |   9 |   1 |         0 |     0
 10 |     1010 |  1 |   5 |   1 |         1 |     1
 11 |     1011 |  0 |  11 |   3 |         1 |     1
 12 |     1100 |  2 |   3 |   3 |         0 |     0
 13 |     1101 |  0 |  13 |   1 |         0 |     0
 14 |     1110 |  1 |   7 |   3 |         0 |     0
 15 |     1111 |  0 |  15 |   3 |         1 |     1


In [16]:
class SolutionBit:
    def findKthBit(self, n: int, k: int) -> str:
        low = k & -k
        m = k // low                         # phần lẻ của k
        odd_k = low == 1
        return "1" if ((m % 4 == 3) == odd_k) else "0"

print([SolutionBit().findKthBit(n, k) for n, k, _ in examples + more])

['0', '1', '0', '1']


**Độ phức tạp:** Time `O(1)` (phép toán trên số ≤ `2²⁰`), Space `O(1)`.

## 8. TRACE từng bước

Trace cách 3 với `n = 5, k = 20` (`len(S₅) = 31`):

| bước | n | mid | k | so sánh | thao tác | flip |
|------|---|-----|---|---------|----------|------|
| 1 | 5 | 16 | 20 | k > mid | k = 32 − 20 = 12, flip ^= 1 | 1 |
| 2 | 4 | 8 | 12 | k > mid | k = 16 − 12 = 4, flip ^= 1 | 0 |
| 3 | 3 | 4 | 4 | k = mid | trả `1 ^ flip` = `1` | 0 |

Hàm `trace_iter` in bảng trên; ta đối chiếu với chuỗi `S₅` thật.

In [17]:
def trace_iter(n, k):
    flip, step = 0, 0
    print(f"{'bước':>4} | {'n':>2} | {'mid':>5} | {'k':>6} | thao tác")
    while n > 1:
        step += 1
        mid = 1 << (n - 1)
        if k == mid:
            print(f"{step:>4} | {n:>2} | {mid:>5} | {k:>6} | k = mid -> bit gốc 1, flip={flip} -> {1 ^ flip}")
            return str(1 ^ flip)
        if k > mid:
            nk = (1 << n) - k
            flip ^= 1
            print(f"{step:>4} | {n:>2} | {mid:>5} | {k:>6} | k > mid -> k = {1 << n} - {k} = {nk}, flip={flip}")
            k = nk
        else:
            print(f"{step:>4} | {n:>2} | {mid:>5} | {k:>6} | k < mid -> giữ k")
        n -= 1
    print(f"tới S1: bit gốc 0, flip={flip} -> {flip}")
    return str(flip)

ans = trace_iter(5, 20)
print("Kết quả:", ans, "| chuỗi thật S5[19] =", build_S(5)[19])
print()
trace_iter(4, 11)

bước |  n |   mid |      k | thao tác
   1 |  5 |    16 |     20 | k > mid -> k = 32 - 20 = 12, flip=1
   2 |  4 |     8 |     12 | k > mid -> k = 16 - 12 = 4, flip=0
   3 |  3 |     4 |      4 | k = mid -> bit gốc 1, flip=0 -> 1
Kết quả: 1 | chuỗi thật S5[19] = 1

bước |  n |   mid |      k | thao tác
   1 |  4 |     8 |     11 | k > mid -> k = 16 - 11 = 5, flip=1
   2 |  3 |     4 |      5 | k > mid -> k = 8 - 5 = 3, flip=0
   3 |  2 |     2 |      3 | k > mid -> k = 4 - 3 = 1, flip=1
tới S1: bit gốc 0, flip=1 -> 1


'1'

Đường đi của **mọi** vị trí trong `S₃` (mỗi mũi tên là một bước soi gương hoặc giữ nguyên; `*` đánh dấu có đảo bit):

In [18]:
def path(n, k):
    steps, flip = [], 0
    while n > 1:
        mid = 1 << (n - 1)
        if k == mid:
            steps.append(f"S{n}[{k}]=giữa")
            return " -> ".join(steps), str(1 ^ flip)
        if k > mid:
            steps.append(f"S{n}[{k}]*")
            k, flip = (1 << n) - k, flip ^ 1
        else:
            steps.append(f"S{n}[{k}]")
        n -= 1
    steps.append("S1[1]='0'")
    return " -> ".join(steps), str(flip)

S3 = build_S(3)
for k in range(1, 8):
    p, b = path(3, k)
    print(f"k={k}: {p:40s} => {b} (thật: {S3[k - 1]})")

k=1: S3[1] -> S2[1] -> S1[1]='0'              => 0 (thật: 0)
k=2: S3[2] -> S2[2]=giữa                      => 1 (thật: 1)
k=3: S3[3] -> S2[3]* -> S1[1]='0'             => 1 (thật: 1)
k=4: S3[4]=giữa                               => 1 (thật: 1)
k=5: S3[5]* -> S2[3]* -> S1[1]='0'            => 0 (thật: 0)
k=6: S3[6]* -> S2[2]=giữa                     => 0 (thật: 0)
k=7: S3[7]* -> S2[1] -> S1[1]='0'             => 1 (thật: 1)


In [19]:
def trace_rec(n, k, depth=0):
    pad = "  " * depth
    if n == 1:
        print(f"{pad}f(1, {k}) = '0' (cơ sở)")
        return "0"
    mid = 1 << (n - 1)
    if k == mid:
        print(f"{pad}f({n}, {k}): k = mid -> '1'")
        return "1"
    if k < mid:
        print(f"{pad}f({n}, {k}): nửa trái -> f({n - 1}, {k})")
        return trace_rec(n - 1, k, depth + 1)
    print(f"{pad}f({n}, {k}): nửa phải -> đảo f({n - 1}, {(1 << n) - k})")
    b = trace_rec(n - 1, (1 << n) - k, depth + 1)
    r = "1" if b == "0" else "0"
    print(f"{pad}=> đảo '{b}' thành '{r}'")
    return r

trace_rec(5, 22)

f(5, 22): nửa phải -> đảo f(4, 10)
  f(4, 10): nửa phải -> đảo f(3, 6)
    f(3, 6): nửa phải -> đảo f(2, 2)
      f(2, 2): k = mid -> '1'
    => đảo '1' thành '0'
  => đảo '0' thành '1'
=> đảo '1' thành '0'


'0'

## 9. Code nộp LeetCode

Bản nộp chính — cách 3 (vòng lặp, dễ giải thích, `O(n)`/`O(1)`), đúng chữ ký `findKthBit(self, n: int, k: int) -> str`:

In [20]:
class Solution:
    def findKthBit(self, n: int, k: int) -> str:
        flip = 0                            # số lần đảo bit (mod 2)
        while n > 1:
            mid = 1 << (n - 1)              # vị trí "1" ở giữa S_n
            if k == mid:
                return "1" if flip == 0 else "0"
            if k > mid:                     # nửa phải = gương đảo bit của S_{n-1}
                k = (1 << n) - k
                flip ^= 1
            n -= 1
        return "0" if flip == 0 else "1"   # S_1 = "0"

for n, k, want in examples + more:
    got = Solution().findKthBit(n, k)
    print(n, k, "->", got, "OK" if got == want else "SAI")

3 1 -> 0 OK
4 11 -> 1 OK
1 1 -> 0 OK
2 3 -> 1 OK


## 10. Vì sao thuật toán đúng

**Cấu trúc của `Sₙ`.** Theo định nghĩa, với `L = 2ⁿ⁻¹ − 1 = len(Sₙ₋₁)`:
- vị trí `1..L` là `Sₙ₋₁`;
- vị trí `L + 1 = 2ⁿ⁻¹` là `"1"`;
- vị trí `L + 1 + j` (`1 ≤ j ≤ L`) là `invert(Sₙ₋₁)[L − j + 1]` (do đảo ngược).

**Ba trường hợp.** `k < mid` ⇒ bit = `Sₙ₋₁[k]`. `k = mid` ⇒ `"1"`. `k > mid` ⇒ với `j = k − mid`, bit = `invert(Sₙ₋₁[L − j + 1]) = invert(Sₙ₋₁[2ⁿ − k])` (tính ở mục 6). Vị trí mới `2ⁿ − k ∈ [1, L]` hợp lệ.

**Invariant của vòng lặp:** "đáp án = `Sₙ[k]` đảo `flip` lần". Ban đầu `flip = 0` đúng. Mỗi vòng giữ nguyên invariant theo ba trường hợp trên (trường hợp gương: `Sₙ[k] = invert(Sₙ₋₁[k'])` nên tăng `flip`). Vòng lặp dừng khi gặp giữa hoặc `n = 1` (`S₁ = "0"`), lúc đó áp dụng `flip` cho bit gốc ⇒ đúng. Vòng lặp chạy tối đa `n − 1` lần ⇒ kết thúc.

## 11. Phân tích Time / Space

| Cách | Time | Space | Ghi chú |
|------|------|-------|---------|
| 1. Sinh chuỗi | `O(2ⁿ)` | `O(2ⁿ)` | `n ≤ 20` vẫn chạy được (~1 triệu ký tự) |
| 2. Đệ quy theo vị trí | `O(n)` | `O(n)` stack | |
| 3. Vòng lặp + `flip` | `O(n)` | `O(1)` | **bản nộp** |
| 4. Công thức bit | `O(1)` | `O(1)` | không phụ thuộc `n` |

Nếu phải trả lời `q` truy vấn với cùng `n` nhỏ, cách 1 sinh chuỗi **một lần** rồi tra `O(1)` cũng là lựa chọn hợp lý (`O(2ⁿ + q)`).

## 12. Bộ test

Đối chiếu **mọi** `(n, k)` với `n ≤ 14` (tổng ~32 nghìn vị trí) với chuỗi sinh thật; random `(n, k)` với `n` tới 20 (seed cố định); các vị trí đặc biệt (đầu, giữa, cuối, lũy thừa của 2); đo thời gian.

In [21]:
METHODS = {
    "rec": rec,
    "SolutionRec": lambda n, k: SolutionRec().findKthBit(n, k),
    "iterative": iterative,
    "formula": formula,
    "SolutionBit": lambda n, k: SolutionBit().findKthBit(n, k),
    "Solution": lambda n, k: Solution().findKthBit(n, k),
}
for n, k, want in examples + more:
    assert brute(n, k) == want
    for name, f in METHODS.items():
        assert f(n, k) == want, (name, n, k)
print("ví dụ: OK")

cnt = 0
for n in range(1, 15):
    s = build_S(n)
    for k in range(1, len(s) + 1):
        for name, f in METHODS.items():
            assert f(n, k) == s[k - 1], (name, n, k)
        cnt += 1
print(f"Mọi (n, k) với n ≤ 14 ({cnt} vị trí): mọi cách khớp chuỗi sinh thật")

ví dụ: OK


Mọi (n, k) với n ≤ 14 (32752 vị trí): mọi cách khớp chuỗi sinh thật


In [22]:
rng = random.Random(1545)
for _ in range(3000):
    n = rng.randint(1, 20)
    k = rng.randint(1, (1 << n) - 1)
    ref = s20[k - 1]                        # S_n là tiền tố của S_20
    for name, f in METHODS.items():
        assert f(n, k) == ref, (name, n, k)
special = [(20, 1), (20, (1 << 20) - 1), (20, 1 << 19), (20, (1 << 19) + 1), (20, (1 << 19) - 1)]
for n, k in special:
    assert all(f(n, k) == s20[k - 1] for f in METHODS.values())
print("3000 (n, k) ngẫu nhiên tới n = 20 (seed 1545) + vị trí đặc biệt: OK")

3000 (n, k) ngẫu nhiên tới n = 20 (seed 1545) + vị trí đặc biệt: OK


In [23]:
for name in ["rec", "iterative", "formula", "Solution"]:
    f = METHODS[name]
    t0 = time.perf_counter()
    for k in range(1, 200001):
        f(20, k)
    print(f"{name:9s}: 200000 truy vấn n=20 trong {time.perf_counter() - t0:.3f}s")

rec      : 200000 truy vấn n=20 trong 0.435s


iterative: 200000 truy vấn n=20 trong 0.384s
formula  : 200000 truy vấn n=20 trong 0.041s


Solution : 200000 truy vấn n=20 trong 0.380s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên đảo bit khi soi gương

Nửa phải là `reverse(invert(...))`, nếu chỉ soi gương mà không đảo bit thì sai.

In [24]:
def wrong_no_invert(n, k):
    while n > 1:
        mid = 1 << (n - 1)
        if k == mid:
            return "1"
        if k > mid:
            k = (1 << n) - k                # SAI: thiếu đảo bit
        n -= 1
    return "0"

for n, k, want in examples + [(3, 5, "0"), (3, 7, "1")]:
    print(n, k, "| sai:", wrong_no_invert(n, k), "| đúng:", Solution().findKthBit(n, k))

3 1 | sai: 0 | đúng: 0
4 11 | sai: 0 | đúng: 1
3 5 | sai: 0 | đúng: 0
3 7 | sai: 0 | đúng: 1


Sửa: `flip ^= 1` mỗi lần soi gương.

### Lỗi 2 — Chỉ số gương lệch một (`2ⁿ − k + 1` hoặc `2ⁿ − 1 − k`)

Tổng hai vị trí đối xứng qua giữa là `2·mid = 2ⁿ`; các công thức "na ná" đều lệch.

In [25]:
def wrong_mirror(n, k):
    flip = 0
    while n > 1:
        mid = 1 << (n - 1)
        if k == mid:
            return str(1 ^ flip)
        if k > mid:
            k = (1 << n) - 1 - k            # SAI: phải là 2^n - k
            flip ^= 1
        n -= 1
    return str(flip)

s5 = build_S(5)
bad = [(5, k) for k in range(1, 32) if wrong_mirror(5, k) != s5[k - 1]]
print("Số vị trí sai trong S5:", len(bad), "| ví dụ:", [(n, k, wrong_mirror(n, k), s5[k - 1]) for n, k in bad[:4]])

Số vị trí sai trong S5: 7 | ví dụ: [(5, 6, '1', '0'), (5, 11, '0', '1'), (5, 14, '1', '0'), (5, 22, '1', '0')]


Sửa: `k = (1 << n) − k`. Kiểm tra nhanh: `k = mid + 1` phải ánh xạ về `mid − 1` (vị trí cuối của `Sₙ₋₁`).

### Lỗi 3 — Sinh chuỗi nhưng quên `reverse`

`Sᵢ = Sᵢ₋₁ + "1" + invert(Sᵢ₋₁)` (thiếu đảo ngược) cho chuỗi khác hẳn từ `S₃`.

In [26]:
def wrong_build(n):
    s = "0"
    for _ in range(n - 1):
        s = s + "1" + s.translate(INV)          # SAI: thiếu [::-1]
    return s

for n in range(2, 5):
    print(f"S{n} sai: {wrong_build(n)} | đúng: {build_S(n)}")
print("n=4, k=11 -> sai:", wrong_build(4)[10], "| đúng:", "1")

S2 sai: 011 | đúng: 011
S3 sai: 0111100 | đúng: 0111001
S4 sai: 011110011000011 | đúng: 011100110110001
n=4, k=11 -> sai: 0 | đúng: 1


Sửa: `s + "1" + s.translate(INV)[::-1]`.

### Lỗi 4 — Trả về số nguyên thay vì chuỗi

Đề yêu cầu `str` (`"0"`/`"1"`). Trả `0`/`1` sẽ bị chấm sai vì `0 != "0"`.

In [27]:
def wrong_int(n, k):
    flip = 0
    while n > 1:
        mid = 1 << (n - 1)
        if k == mid:
            return 1 ^ flip                  # SAI: kiểu int
        if k > mid:
            k = (1 << n) - k
            flip ^= 1
        n -= 1
    return flip

for n, k, want in examples:
    got = wrong_int(n, k)
    print(f"n={n}, k={k}: trả {got!r} -> so với {want!r}: {got == want}")

n=3, k=1: trả 0 -> so với '0': False
n=4, k=11: trả 1 -> so với '1': False


Sửa: bọc `str(...)` hoặc trả trực tiếp `"0"`/`"1"`.

### Lỗi 5 — Đảo bit bằng hai lần `replace`

`s.replace("0", "1").replace("1", "0")` biến **mọi** ký tự thành `"0"` (lần thay thứ hai nuốt luôn các `"1"` vừa tạo).

In [28]:
def wrong_invert(s):
    return s.replace("0", "1").replace("1", "0")    # SAI

def build_with(inv, n):
    s = "0"
    for _ in range(n - 1):
        s = s + "1" + inv(s)[::-1]
    return s

print("invert sai của '0111001':", wrong_invert("0111001"))
print("S4 sai:", build_with(wrong_invert, 4), "| đúng:", build_S(4))

invert sai của '0111001': 0000000
S4 sai: 010100010000000 | đúng: 011100110110001


Sửa: `s.translate(str.maketrans("01", "10"))`, hoặc thay qua ký tự trung gian (`"0"→"x"`, `"1"→"0"`, `"x"→"1"`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Chuỗi/dãy định nghĩa đệ quy kiểu "nửa trái + giữa + biến đổi(nửa trái)"** ⇒ truy ngược vị trí, không sinh chuỗi: mỗi bước giảm `n` một đơn vị.
- Biến đổi "đảo bit" ⇒ chỉ cần **tính chẵn lẻ** số lần đảo (`flip ^= 1`).
- Biến đổi "đảo ngược" ⇒ vị trí gương: `k' = (tổng hai đầu) − k`.
- Thấy độ dài `2ⁿ − 1` ⇒ nghĩ tới vị trí giữa `2ⁿ⁻¹`, thao tác bit, và quan hệ với cây nhị phân đầy đủ.
- Bài cùng họ: *K-th Symbol in Grammar* (779), *Elimination Game* (390), *Find the Kth Character in String Game* (3304), *Regular paperfolding sequence* (dãy gấp giấy — chính là dạng của bài này).

## 15. Biến thể của bài

1. **`n` rất lớn** (ví dụ `n ≤ 60`, `k ≤ 10¹⁸`): cách 1 chết, cách 3/4 vẫn chạy ngay (Python có số nguyên lớn).
2. **Đếm số bit 1 trong `Sₙ`**: `ones(n) = ones(n−1) + 1 + (len(n−1) − ones(n−1)) = len(n−1) + 1 = 2ⁿ⁻¹` với `n ≥ 2` (riêng `S₁ = "0"` có 0 bit 1).
3. **Đếm số bit 1 trong `k` ký tự đầu**: đệ quy theo vị trí tương tự (bài tập 5).
4. **Không đảo bit, chỉ đảo ngược** (`Sᵢ = Sᵢ₋₁ + "1" + reverse(Sᵢ₋₁)`): bỏ `flip`.
5. **Giữa là `"0"`** hoặc **`S₁` khác**: chỉ thay hai giá trị gốc trong vòng lặp.

In [29]:
def variant_big(n, k):
    """Biến thể 1: n tới 60, dùng đúng vòng lặp của Solution."""
    return Solution().findKthBit(n, k)

k_big = 10**18 + 7
print("n=60, k=10^18+7 ->", variant_big(60, k_big), "| công thức:", formula(60, k_big))
print("Biến thể 2: số bit 1 của S_n (n=1..8):", [build_S(n).count("1") for n in range(1, 9)], "-> 2^(n-1) với n >= 2, riêng S1 là 0")

n=60, k=10^18+7 -> 1 | công thức: 1
Biến thể 2: số bit 1 của S_n (n=1..8): [0, 2, 4, 8, 16, 32, 64, 128] -> 2^(n-1) với n >= 2, riêng S1 là 0


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Sinh chuỗi | làm theo định nghĩa | `O(2ⁿ)` | `O(2ⁿ)` | `n` nhỏ, nhiều truy vấn cùng `n` |
| 2. Đệ quy vị trí | 3 trường hợp: trái / giữa / phải | `O(n)` | `O(n)` | dễ viết từ định nghĩa |
| 3. Vòng lặp + `flip` | khử đệ quy, nhớ số lần đảo | `O(n)` | `O(1)` | **bản nộp** |
| 4. Công thức bit | `k = 2ᵃ·m`, xét `m mod 4` | `O(1)` | `O(1)` | `n` cực lớn / thi đấu |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `findKthBit(4, 8)` và `findKthBit(4, 12)` trả về gì?

**Bài 2 (tính tay):** Tính `findKthBit(5, 20)` bằng cách đi ngược vị trí (ghi lại `k` và `flip` mỗi bước).

**Bài 3 (điền TODO):** Hoàn thiện `ex3_iter(n, k)` (cách 3).

**Bài 4 (tự viết lại):** Viết `ex4_build(n)` sinh `Sₙ` **dạng list số nguyên 0/1** (không dùng chuỗi), kiểm tra khớp `build_S`.

**Bài 5 (mini challenge):** Viết `ex5_prefix_ones(n, k)` = số bit 1 trong `k` ký tự đầu của `Sₙ`, trong `O(n)` (không sinh chuỗi).

In [30]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_iter(n: int, k: int) -> str:
    flip = 0
    while n > 1:
        mid = 1 << (n - 1)
        # TODO: nếu k == mid trả str(1 ^ flip)
        # TODO: nếu k > mid: k = 2^n - k, flip ^= 1
        n -= 1
    return str(flip)

In [31]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_build(n: int) -> List[int]:
    # TODO
    raise NotImplementedError

def ex5_prefix_ones(n: int, k: int) -> int:
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `n = 4`: `mid = 8`. (2) `k = 8` ⇒ giữa ⇒ `"1"`. (3) `k = 12 > 8` ⇒ `k' = 4` ở `S₃`, `mid = 4` ⇒ `"1"`, đảo ⇒ `"0"`.

**Bài 2.** (1) `n = 5`, `mid = 16`, `k = 20 > 16` ⇒ `k = 12`, `flip = 1`. (2) `n = 4`, `mid = 8`, `k = 12 > 8` ⇒ `k = 4`, `flip = 0`. (3) `n = 3`, `mid = 4 = k` ⇒ `"1"`.

**Bài 3.** (1) `if k == mid: return str(1 ^ flip)`. (2) `if k > mid: k = (1 << n) - k; flip ^= 1`. (3) Sau vòng lặp, bit gốc là `0`.

**Bài 4.** (1) Bắt đầu `s = [0]`. (2) Mỗi bước: `s = s + [1] + [1 - b for b in reversed(s)]`. (3) So `"".join(map(str, s)) == build_S(n)`.

**Bài 5.** (1) `ones(m) = 2ᵐ⁻¹` cho toàn bộ `Sₘ` khi `m ≥ 2` (`ones(1) = 0`), `len(m) = 2ᵐ − 1`. (2) `k < mid` ⇒ đệ quy trái; `k = mid` ⇒ `ones(n−1) + 1`. (3) `k > mid`: phần phải dài `j = k − mid` là gương đảo của **`j` ký tự cuối** `Sₙ₋₁`; số bit 1 trong đó = `j − (ones(n−1) − prefix(n−1, L − j))` với `L = 2ⁿ⁻¹ − 1`.

## 19. Đáp án tham khảo cho bài tập

In [32]:
# Bài 1
a1 = (Solution().findKthBit(4, 8), Solution().findKthBit(4, 12))
print("Bài 1:", a1)
assert a1 == ("1", "0") and build_S(4)[7] == "1" and build_S(4)[11] == "0"

# Bài 2
a2 = trace_iter(5, 20)
assert a2 == "1" == build_S(5)[19]

Bài 1: ('1', '0')
bước |  n |   mid |      k | thao tác
   1 |  5 |    16 |     20 | k > mid -> k = 32 - 20 = 12, flip=1
   2 |  4 |     8 |     12 | k > mid -> k = 16 - 12 = 4, flip=0
   3 |  3 |     4 |      4 | k = mid -> bit gốc 1, flip=0 -> 1


In [33]:
# Bài 3
def ex3_iter_sol(n, k):
    flip = 0
    while n > 1:
        mid = 1 << (n - 1)
        if k == mid:
            return str(1 ^ flip)
        if k > mid:
            k = (1 << n) - k
            flip ^= 1
        n -= 1
    return str(flip)

for n in range(1, 12):
    s = build_S(n)
    assert all(ex3_iter_sol(n, k) == s[k - 1] for k in range(1, len(s) + 1))
print("Bài 3: OK")

Bài 3: OK


In [34]:
# Bài 4
def ex4_build_sol(n):
    s = [0]
    for _ in range(n - 1):
        s = s + [1] + [1 - b for b in reversed(s)]
    return s

for n in range(1, 15):
    assert "".join(map(str, ex4_build_sol(n))) == build_S(n)
print("Bài 4: OK, S3 =", ex4_build_sol(3))

Bài 4: OK, S3 = [0, 1, 1, 1, 0, 0, 1]


In [35]:
# Bài 5
def ex5_prefix_ones_sol(n, k):
    if k == 0:
        return 0
    if n == 1:
        return 0                               # S1 = "0"
    mid = 1 << (n - 1)
    L = mid - 1                                # len(S_{n-1})
    ones_prev = (1 << (n - 2)) if n > 2 else 0 # số bit 1 của S_{n-1} (S1 = "0" không có bit 1)
    if k < mid:
        return ex5_prefix_ones_sol(n - 1, k)
    if k == mid:
        return ones_prev + 1
    j = k - mid
    ones_last_j = ones_prev - ex5_prefix_ones_sol(n - 1, L - j)   # bit 1 trong j ký tự cuối S_{n-1}
    return ones_prev + 1 + (j - ones_last_j)

for n in range(1, 13):
    s = build_S(n)
    for k in range(0, len(s) + 1):
        assert ex5_prefix_ones_sol(n, k) == s[:k].count("1"), (n, k)
print("Bài 5: OK, ví dụ prefix_ones(20, 10^6) =", ex5_prefix_ones_sol(20, 10**6), "| kiểm tra:", s20[:10**6].count("1"))

Bài 5: OK, ví dụ prefix_ones(20, 10^6) = 500004 | kiểm tra: 500004


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def findKthBit(self, n, k):
        flip = 0
        while n > 1:
            mid = 1 << (n - 1)
            if k == mid: return str(1 ^ flip)   # giữa luôn là "1"
            if k > mid:                         # nửa phải: gương + đảo bit
                k = (1 << n) - k
                flip ^= 1
            n -= 1
        return str(flip)                        # S1 = "0"
```

**Template "dãy tự đồng dạng: trái + giữa + biến đổi(trái)":** tìm vị trí giữa; trái → giữ nguyên; giữa → giá trị cố định; phải → ánh xạ gương (`tổng − k`) và ghi nhận biến đổi (đảo bit ⇒ `flip ^= 1`).

**Checklist trước khi submit:**
- [ ] `mid = 2ⁿ⁻¹`, vị trí gương `2ⁿ − k` (không lệch một).
- [ ] Mỗi lần soi gương đều đảo bit.
- [ ] Trả về **chuỗi** `"0"`/`"1"`.
- [ ] Nếu sinh chuỗi: đảo bit bằng `translate`, nhớ `[::-1]`.
- [ ] Thử nhanh: `(3,1) → "0"`, `(4,11) → "1"`, `(1,1) → "0"`, `(2,3) → "1"`.